# LIVE SHOWCASE!!!!!

# 1. imports

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import scipy.signal
import matplotlib.pyplot as plt
import joblib

import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, Model
from tensorflow.keras.models import load_model

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix

from librosa.feature import rms

# 2. loading....

In [31]:
model = load_model("CNN model/best_model_1d_cnn.keras")
scaler = joblib.load("CNN model/scaler_lstm_experiment.pkl")
le = joblib.load("CNN model/label_encoder_lstm_experiment.pkl")

In [8]:
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print("YAMNet geladen.")

YAMNet geladen.


# 3. Code van andere files

In [9]:
def extract_embeddings_sequence(waveform):
    scores, embeddings, spectrogram = yamnet_model(waveform)
    return embeddings.numpy()

In [10]:
def pad_audio(y, target_length):
    """Pads the audio with zeros to match the target length."""
    if len(y) < target_length:
        padding = target_length - len(y)
        y = np.pad(y, (0, padding), 'constant')
    return y

def truncate_audio(y, target_length):
    """Truncates the audio to match the target length."""
    if len(y) > target_length:
        y = y[:target_length]
    return y

def adjust_audio(y, target_length=32000):
    """Adjusts the audio to match the target length by padding or truncating."""
    if len(y) < target_length:
        return pad_audio(y, target_length)
    else:
        return truncate_audio(y, target_length)

def ensure_sample_rate(original_sample_rate, waveform,
                       desired_sample_rate=16000):
    """Resample waveform if required."""
    if original_sample_rate != desired_sample_rate:
        desired_length = int(round(float(len(waveform)) /
                                  original_sample_rate * desired_sample_rate))
        waveform = scipy.signal.resample(waveform, desired_length)
    return desired_sample_rate, waveform

In [11]:
def segment_cough_sound(signal, sr, cough_threshold=0.05, min_cough_duration=0.1, padding=0.05):

    hop_length = int(min_cough_duration*sr)
    if len(signal.shape) > 1:
        signal = np.mean(signal, axis=1)

    energy = rms(y=signal, hop_length=hop_length)[0]

    # Normalize the energy values
    normalized_energy = (energy - np.min(energy)) / (np.max(energy) - np.min(energy))

    # Set the energy threshold for event detection
    cough_threshold = np.max(normalized_energy) * cough_threshold
    min_cough_samples = round(sr * min_cough_duration)


    # Find the cough segments
    cough_segments = []
    event_start = None

    for i, value in enumerate(normalized_energy):
        if value >= cough_threshold:
            if event_start is None:
                event_start = i*hop_length
        else:
            if event_start is not None:
                cough_duration = i*hop_length - event_start
                if cough_duration >= min_cough_samples:
                    event_end = i*hop_length + int(padding * sr)
                    event_start -= int(padding * sr)
                    event_start = max(event_start, 0)
                    cough_segments.append(signal[event_start: event_end+1])
                event_start = None

    # Convert cough segments to time in seconds
    # cough_segments = [(start / sr, end / sr) for start, end in cough_segments]

    return cough_segments


def process_cough_file(file_path, save_segments=True):
    new_path = file_path.replace(".wav", "_segmented")

    if os.path.exists(new_path) and os.listdir(new_path):
        print(f"Skipping (already segmented): {file_path}")
        return

    try:
        signal, sr = sf.read(file_path)

        if len(signal) == 0 or sr == 0:
            print(f"Skipping (invalid file): {file_path}")
            return

        if len(signal.shape) > 1:
            signal = np.mean(signal, axis=1)

        segments = segment_cough_sound(signal, sr)

        print(f"{file_path} → {len(segments)} segments found")

        if save_segments:
            os.makedirs(new_path, exist_ok=True)
            for i, seg in enumerate(segments):
                seg_path = os.path.join(new_path, f"seg{i}.wav")
                sf.write(seg_path, seg, sr)

    except Exception as e:
        print(f"Error processing {file_path}: {e}")





def process_audio_file(file_path, target_length=32000, desired_sr=16000):
    new_path = file_path.replace(".wav", "_processed.wav")

    if os.path.exists(new_path):
        print(f"Skipping (already processed): {new_path}")
        return

    try:
        waveform, sr = sf.read(file_path)

        if len(waveform) == 0 or sr == 0:
            print(f"Skipping (invalid file): {file_path}")
            return

        # Convert stereo → mono if needed
        if len(waveform.shape) > 1:
            waveform = np.mean(waveform, axis=1)

        # Resample
        sr, waveform = ensure_sample_rate(sr, waveform, desired_sr)

        # Pad / truncate
        waveform = adjust_audio(waveform, target_length)

        # Save processed file
        sf.write(new_path, waveform, sr)
        print(f"Saved: {new_path}")

    except Exception as e:
        print(f"Error processing {file_path}: {e}")

In [39]:
def test_on_person(file_path, metadata):
    process_cough_file(f"hoestjes/{file_path}.wav")
    process_audio_file(f"hoestjes/{file_path}_segmented/seg0.wav")


    waveform, _ = librosa.load(f"hoestjes/{file_path}_segmented/seg0_processed.wav", sr=16000)

    emb = extract_embeddings_sequence(waveform) 

    emb = emb[np.newaxis, ...] 

    metadata_scaled = scaler.transform(metadata)

    pred = model.predict([emb, metadata_scaled], verbose=0)
    klasse = le.inverse_transform([np.argmax(pred)])[0]
    print(f"Voorspelling: {klasse}")
    print(f"Kansen per klasse: {dict(zip(le.classes_, pred[0].round(3)))}")

# 4. Tijd om te testen

In [32]:
# Volgorde: age, gender, tbContactHistory, wheezingHistory, phlegmCough, familyAsthmaHistory, feverHistory, coldPresent, packYears, coldPresent_missing
meta = np.array([[22, 0, 0, 0, 0, 1, 0, 0, 15, 0]], dtype=np.float32)

test_on_person("Jonas", meta)

Skipping (already segmented): hoestjes/Jonas.wav
Skipping (already processed): hoestjes/Jonas_segmented/seg0_processed.wav
Voorspelling: 0
Kansen per klasse: {np.int64(0): np.float32(0.979), np.int64(1): np.float32(0.021), np.int64(2): np.float32(0.0)}


In [33]:
# Volgorde: age, gender, tbContactHistory, wheezingHistory, phlegmCough, familyAsthmaHistory, feverHistory, coldPresent, packYears, coldPresent_missing
meta = np.array([[21, 0, 0, 0, 0, 1, 0, 0, 0, 0]], dtype=np.float32)

test_on_person("Briek", meta)

Skipping (already segmented): hoestjes/Briek.wav
Skipping (already processed): hoestjes/Briek_segmented/seg0_processed.wav
Voorspelling: 0
Kansen per klasse: {np.int64(0): np.float32(0.858), np.int64(1): np.float32(0.142), np.int64(2): np.float32(0.0)}


In [34]:
# Volgorde: age, gender, tbContactHistory, wheezingHistory, phlegmCough, familyAsthmaHistory, feverHistory, coldPresent, packYears, coldPresent_missing
meta = np.array([[47, 1, 0, 1, 0, 0, 0, 0, 0, 0]], dtype=np.float32)

test_on_person("Inge", meta)

Skipping (already segmented): hoestjes/Inge.wav
Skipping (already processed): hoestjes/Inge_segmented/seg0_processed.wav
Voorspelling: 1
Kansen per klasse: {np.int64(0): np.float32(0.002), np.int64(1): np.float32(0.954), np.int64(2): np.float32(0.045)}


In [40]:
# Volgorde: age, gender, tbContactHistory, wheezingHistory, phlegmCough, familyAsthmaHistory, feverHistory, coldPresent, packYears, coldPresent_missing
meta = np.array([[17, 1, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=np.float32)

test_on_person("Flore", meta)

Skipping (already segmented): hoestjes/Flore.wav
Skipping (already processed): hoestjes/Flore_segmented/seg0_processed.wav
Voorspelling: 1
Kansen per klasse: {np.int64(0): np.float32(0.438), np.int64(1): np.float32(0.544), np.int64(2): np.float32(0.017)}


In [45]:
# Volgorde: age, gender, tbContactHistory, wheezingHistory, phlegmCough, familyAsthmaHistory, feverHistory, coldPresent, packYears, coldPresent_missing
meta = np.array([[43, 0, 0, 1, 1, 1, 0, 0, 800, 0]], dtype=np.float32)

test_on_person("Meneer", meta)

Skipping (already segmented): hoestjes/Meneer.wav
Skipping (already processed): hoestjes/Meneer_segmented/seg0_processed.wav
Voorspelling: 1
Kansen per klasse: {np.int64(0): np.float32(0.0), np.int64(1): np.float32(0.937), np.int64(2): np.float32(0.063)}


In [49]:
# Volgorde: age, gender, tbContactHistory, wheezingHistory, phlegmCough, familyAsthmaHistory, feverHistory, coldPresent, packYears, coldPresent_missing
meta = np.array([[36, 1, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=np.float32)

test_on_person("Mevrouw", meta)

Skipping (already segmented): hoestjes/Mevrouw.wav
Skipping (already processed): hoestjes/Mevrouw_segmented/seg0_processed.wav
Voorspelling: 0
Kansen per klasse: {np.int64(0): np.float32(0.823), np.int64(1): np.float32(0.164), np.int64(2): np.float32(0.013)}
